<a href="https://colab.research.google.com/github/BorysZhyhalo/goit-rdb-hw-07/blob/main/hw_07_rdb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework 7 — Time Series and Semi-Structured Data for ML

## 1. Setup and NYC Taxi Data Loading

In [9]:
!pip install -q pgserver==0.1.4 \
    --python-version 3.12 \
    --only-binary=:all: \
    --target=/usr/local/lib/python3.13/dist-packages

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 73.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 10.5 MB/s eta 0:00:00


In [10]:
%pip install -q "psycopg[binary]" sqlalchemy pandas pyarrow requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.5/215.5 kB 4.4 MB/s eta 0:00:00


In [11]:
import os
import sys
import pandas as pd
import pgserver

from urllib.request import urlretrieve
from sqlalchemy import create_engine, text

print("Python:", sys.version)

pg = pgserver.get_server('/tmp/hw7_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar_one())

/usr/local/lib/python3.13/dist-packages/platformdirs/api.py:459: RuntimeDirWarning: XDG_RUNTIME_DIR is not set, falling back to /tmp/runtime-0
  return Path(self.user_runtime_dir)


Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


## 1.1 NYC Taxi Data Loading

Dataset: NYC TLC Yellow Taxi Trip Records, March 2024.
Source: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page

In [12]:
URL = (
    "https://d37ci6vzurychx.cloudfront.net/trip-data/"
    "yellow_tripdata_2024-03.parquet"
)

parquet_path = "/content/yellow_tripdata_2024-03.parquet"

urlretrieve(URL, parquet_path)

columns = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "trip_distance",
    "fare_amount",
    "passenger_count",
]

df = pd.read_parquet(parquet_path, columns=columns).head(500_000)

print("Loaded rows:", len(df))
print("Shape:", df.shape)

display(df.head())

Loaded rows: 500000
Shape: (500000, 7)


,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,trip_distance,fare_amount,passenger_count
0,2024-03-01 00:18:51,2024-03-01 00:23:45,142,239,1.30,8.6,0.0
1,2024-03-01 00:26:00,2024-03-01 00:29:06,238,24,1.10,7.2,0.0
2,2024-03-01 00:09:22,2024-03-01 00:15:24,263,75,0.86,7.9,1.0
3,2024-03-01 00:33:45,2024-03-01 00:39:34,164,162,0.82,7.9,1.0
4,2024-03-01 00:05:43,2024-03-01 00:26:22,263,7,4.90,25.4,0.0


In [13]:
rows_before = len(df)

df = df[
    (df["fare_amount"] > 0) &
    (df["fare_amount"] < 500) &
    (df["trip_distance"] > 0) &
    (df["trip_distance"] < 100) &
    (df["PULocationID"].notna())
].copy()

print("Before cleaning:", rows_before)
print("After cleaning:", len(df))
print("Removed rows:", rows_before - len(df))

Before cleaning: 500000
After cleaning: 486964
Removed rows: 13036


In [14]:
df.to_sql(
    "nyc_taxi_raw",
    engine,
    if_exists="replace",
    index=False,
    chunksize=10_000,
)

pd.read_sql("""
    SELECT COUNT(*) AS n_rows
    FROM nyc_taxi_raw;
""", engine)

,n_rows
0,486964


## 1.2 Daily Time-Series Aggregation

Aggregate NYC Taxi trips by pickup zone and local NYC calendar day.

In [15]:
with engine.begin() as conn:
    conn.execute(text("""
        DROP TABLE IF EXISTS t7_hw_taxi_daily;

        CREATE TABLE t7_hw_taxi_daily AS
        SELECT
            "PULocationID"::INTEGER AS pickup_zone_id,
            DATE_TRUNC('day', tpep_pickup_datetime::TIMESTAMP)::DATE AS ts_day,
            COUNT(*)::INTEGER AS trip_count,
            AVG(fare_amount)::NUMERIC(10, 2) AS avg_fare,
            SUM(trip_distance)::NUMERIC(12, 2) AS total_distance
        FROM nyc_taxi_raw
        WHERE fare_amount > 0
          AND fare_amount < 500
          AND trip_distance > 0
          AND trip_distance < 100
          AND "PULocationID" IS NOT NULL
        GROUP BY
            "PULocationID"::INTEGER,
            DATE_TRUNC('day', tpep_pickup_datetime::TIMESTAMP)::DATE;

        ALTER TABLE t7_hw_taxi_daily
            ADD PRIMARY KEY (pickup_zone_id, ts_day);
    """))

print("Daily aggregation created successfully.")

Daily aggregation created successfully.


In [17]:
display(pd.read_sql("""
    SELECT
        COUNT(*) AS n_rows,
        COUNT(DISTINCT pickup_zone_id) AS n_zones,
        MIN(ts_day) AS first_day,
        MAX(ts_day) AS last_day,
        SUM(trip_count) AS total_trips
    FROM t7_hw_taxi_daily;
""", engine))

display(pd.read_sql("""
    SELECT *
    FROM t7_hw_taxi_daily
    ORDER BY pickup_zone_id, ts_day
    LIMIT 5;
""", engine))

,n_rows,n_zones,first_day,last_day,total_trips
0,1217,240,2024-02-29,2024-03-06,486964


,pickup_zone_id,ts_day,trip_count,avg_fare,total_distance
0,1,2024-03-01,1,105.0,0.04
1,1,2024-03-02,2,95.0,18.43
2,1,2024-03-03,2,51.0,0.09
3,1,2024-03-04,1,40.0,0.01
4,1,2024-03-05,1,120.0,0.03
